In [ ]:
import pandas as pd

# UCI SMS Spam Collection: real personal texts (ham) and generic spam, both used as label 0 (not MLM)
# loaded from the public GitHub repo so this notebook is self-contained (no local files needed)
SMS_URL = "https://raw.githubusercontent.com/adampharrels/isThisMLM/main/data/raw/SMSSpamCollection"
sms = pd.read_csv(SMS_URL, sep="\t", header=None, names=["sms_label", "text"])
sms["label"] = 0
sms["source"] = "sms"
print(sms.shape)
sms["sms_label"].value_counts()

In [ ]:
# Hand-transcribed real MLM recruitment messages/posts from r/antiMLM, label 1
POSITIVES_URL = "https://raw.githubusercontent.com/adampharrels/isThisMLM/main/data/raw/mlm_positives.csv"
pos = pd.read_csv(POSITIVES_URL)
pos["source"] = "reddit"
print(pos.shape)
pos["type"].value_counts()

In [3]:
df = pd.concat([sms, pos], ignore_index=True)
df = df.drop_duplicates(subset="text").reset_index(drop=True)
df["n_words"] = df["text"].str.split().str.len()
print(df["label"].value_counts())
df.groupby("label")["n_words"].describe()

label
0    5169
1     129
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,5169.0,15.439930,11.117073,1.0,7.0,12.0,22.0,171.0
1,129.0,43.178295,44.963779,6.0,17.0,30.0,51.0,297.0


In [4]:
df[df["label"] == 1].sample(10, random_state=0)[["text", "type"]]

,text,type
5209,ARE YOU ROCKSTAR💥 I WANT YOU👆🏽 ON MY TEAM... D...,post
5193,I'd like to know if anyone is interested in a ...,post
5276,You don’t have to join if you don’t want to!,dm
5220,Unlimited Earning Potential: As an Independent...,post
5177,Anyone looking to make extra income from home ...,post
5291,Let me know if you are open to chatting about ...,dm
5213,"People around the world, just like you, are ta...",post
5273,"hi hun it’s been ages, just wondered if you wa...",dm
5242,Join Saladmaster today and embark on an exciti...,post
5176,Would you be interested in a side hustle with ...,dm


In [5]:
df["text"].duplicated().sum()
df[df["text"].duplicated(keep=False)].sort_values("text")



,sms_label,text,label,source,type,source_url,n_words


In [6]:
X = df["text"]
Y = df["label"]

In [7]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, Y)):
    n_train_pos = Y.iloc[train_idx].sum()
    n_test_pos = Y.iloc[test_idx].sum()
    print(f"fold {fold}: train={len(train_idx)} (pos={n_train_pos}), test={len(test_idx)} (pos={n_test_pos})")

fold 0: train=4238 (pos=103), test=1060 (pos=26)
fold 1: train=4238 (pos=103), test=1060 (pos=26)
fold 2: train=4238 (pos=103), test=1060 (pos=26)
fold 3: train=4239 (pos=104), test=1059 (pos=25)
fold 4: train=4239 (pos=103), test=1059 (pos=26)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, average_precision_score

fold_precisions = []
fold_recalls = []
fold_pr_aucs = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, Y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = Y.iloc[train_idx], Y.iloc[test_idx]

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words="english")
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    model.fit(X_train_vec, y_train)

    y_proba = model.predict_proba(X_test_vec)[:, 1]
    y_pred = model.predict(X_test_vec)

    p = precision_score(y_test, y_pred)
    r = recall_score(y_test, y_pred)
    pr_auc = average_precision_score(y_test, y_proba)

    fold_precisions.append(p)
    fold_recalls.append(r)
    fold_pr_aucs.append(pr_auc)

    print(f"fold {fold}: precision={p:.2f} recall={r:.2f} pr_auc={pr_auc:.2f}")

print()
print(f"mean precision: {sum(fold_precisions)/5:.2f}")
print(f"mean recall:    {sum(fold_recalls)/5:.2f}")
print(f"mean pr_auc:    {sum(fold_pr_aucs)/5:.2f}")

fold 0: precision=0.85 recall=0.85 pr_auc=0.90
fold 1: precision=0.85 recall=0.88 pr_auc=0.86
fold 2: precision=0.96 recall=0.85 pr_auc=0.94
fold 3: precision=0.95 recall=0.80 pr_auc=0.95
fold 4: precision=0.87 recall=0.77 pr_auc=0.87

mean precision: 0.90
mean recall:    0.83
mean pr_auc:    0.90


In [9]:
import numpy as np

# fit on ALL the data just to inspect learned words — not a real evaluation
vectorizer_full = TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words="english")
X_full_vec = vectorizer_full.fit_transform(X)

model_full = LogisticRegression(class_weight="balanced", max_iter=1000)
model_full.fit(X_full_vec, Y)

words = vectorizer_full.get_feature_names_out()
coefs = model_full.coef_[0]

order = np.argsort(coefs)
top_mlm_idx = order[::-1][:20]
top_notmlm_idx = order[:20]

print("Top 20 words/phrases pushing toward MLM:")
for i in top_mlm_idx:
    print(f"  {words[i]:30s} {coefs[i]:+.2f}")

print("\nTop 20 words/phrases pushing away from MLM:")
for i in top_notmlm_idx:
    print(f"  {words[i]:30s} {coefs[i]:+.2f}")

Top 20 words/phrases pushing toward MLM:
  business                       +9.31
  income                         +7.46
  join                           +4.51
  financial                      +4.38
  opportunity                    +4.29
  earn                           +4.23
  team                           +4.03
  work                           +3.64
  looking                        +3.48
  freedom                        +3.26
  financial freedom              +3.02
  open                           +2.93
  000                            +2.88
  extra                          +2.86
  company                        +2.82
  hun                            +2.74
  arbonne                        +2.68
  products                       +2.64
  start                          +2.63
  time                           +2.47

Top 20 words/phrases pushing away from MLM:
  ok                             -1.37
  ur                             -1.28
  ll                             -1.07
  got            

In [10]:
from sklearn.metrics import precision_recall_curve

# out-of-fold probabilities: every row scored only by a model that never trained on it
oof_proba = np.zeros(len(Y))

for fold, (train_idx, test_idx) in enumerate(skf.split(X, Y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train = Y.iloc[train_idx]

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words="english")
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    model.fit(X_train_vec, y_train)

    oof_proba[test_idx] = model.predict_proba(X_test_vec)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(Y, oof_proba)

# precision_recall_curve returns one more precision/recall point than thresholds
# (the last point is precision=1, recall=0, with no corresponding threshold)
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    preds_at_t = (oof_proba >= t).astype(int)
    p = precision_score(Y, preds_at_t)
    r = recall_score(Y, preds_at_t)
    print(f"threshold={t:.1f}  precision={p:.2f}  recall={r:.2f}")

threshold=0.1  precision=0.13  recall=0.99
threshold=0.2  precision=0.43  recall=0.95
threshold=0.3  precision=0.66  recall=0.93
threshold=0.4  precision=0.81  recall=0.88
threshold=0.5  precision=0.89  recall=0.83
threshold=0.6  precision=0.93  recall=0.77
threshold=0.7  precision=0.95  recall=0.73
threshold=0.8  precision=0.97  recall=0.67
threshold=0.9  precision=0.97  recall=0.50


In [11]:
THRESHOLD = 0.6

def predict_mlm(text, vectorizer=vectorizer_full, model=model_full, threshold=THRESHOLD, top_k=5):
    x_vec = vectorizer.transform([text])
    proba = model.predict_proba(x_vec)[0, 1]

    feature_names = vectorizer.get_feature_names_out()
    x_arr = x_vec.toarray()[0]
    present_idx = x_arr.nonzero()[0]
    contributions = x_arr[present_idx] * model.coef_[0][present_idx]

    order = contributions.argsort()[::-1]
    top_reasons = [
        (feature_names[present_idx[i]], round(float(contributions[i]), 3))
        for i in order[:top_k]
        if contributions[i] > 0
    ]

    return {
        "probability": round(float(proba), 3),
        "flag": bool(proba >= threshold),
        "top_reasons": top_reasons,
    }

# demo messages — illustrative only, not part of training or evaluation data
demo_messages = [
    "hey! long time no talk, how have you been? so I started this amazing side hustle and I'm looking for a few open minded people to join my team, are you interested in hearing more about a business opportunity?",
    "hey are we still on for dinner saturday? let me know what time works",
    "Hi, I'm reaching out because we have an opening for a Senior Data Analyst role at Acme Corp, salary range $90-110k, would you be open to a quick call this week?",
    "sorry I cant pick the phone right now. pls send a message",
]

for msg in demo_messages:
    result = predict_mlm(msg)
    print(f"text: {msg[:70]}...")
    print(f"  -> probability={result['probability']}, flag={result['flag']}")
    print(f"  -> top reasons: {result['top_reasons']}")
    print()

text: hey! long time no talk, how have you been? so I started this amazing s...
  -> probability=0.997, flag=True
  -> top reasons: [('business', 1.641), ('opportunity', 0.832), ('team', 0.832), ('join', 0.806), ('looking', 0.62)]

text: hey are we still on for dinner saturday? let me know what time works...
  -> probability=0.369, flag=False
  -> top reasons: [('time', 0.643), ('let', 0.577), ('hey', 0.562), ('let know', 0.408), ('works', 0.189)]

text: Hi, I'm reaching out because we have an opening for a Senior Data Anal...
  -> probability=0.131, flag=False
  -> top reasons: [('open', 0.85), ('hi', 0.344)]

text: sorry I cant pick the phone right now. pls send a message...
  -> probability=0.047, flag=False
  -> top reasons: [('right', 0.208), ('message', 0.183), ('phone', 0.149)]



In [12]:
hard_negatives = [
    ("legit recruiter, named role/company",
     "Hi, I'm a recruiter at Google and came across your profile — we have an opening for a Software Engineer role on the Search team, base salary $150k. Would you be open to a 20 minute call this week?"),
    ("friend selling own product, no recruiting",
     "hey! I just started baking custom cakes for orders, would you want one for your birthday? I can do chocolate or vanilla, $35"),
    ("friend selling own craft business",
     "hi! I opened an Etsy shop selling handmade candles, here's the link if you want to check it out, no pressure at all"),
    ("fundraiser invite",
     "Hey, a few of us are doing a charity 5k next month to raise money for the children's hospital, would you want to donate or join the team?"),
    ("job referral from a friend",
     "hey my company is hiring for a marketing coordinator role, thought of you, want me to send you the job posting?"),
    ("event invite",
     "we're hosting a small business owners meetup downtown next Thursday, free entry, just come by if you're around"),
    ("customer-only product pitch, no recruiting",
     "Just wanted to let you know we restocked the lavender candles you liked last time, $18 each, let me know if you want one"),
    ("legit small business announcement",
     "Hi everyone, excited to share that I've opened my own photography business! Booking sessions for the spring, link in bio if interested"),
    ("financial advisor, named firm",
     "This is Sarah from Fidelity, following up on your request for a consultation about your retirement account. Are you free Tuesday at 2pm?"),
    ("volunteer opportunity",
     "Our nonprofit is looking for volunteers for the weekend food drive, no experience needed, would you be interested in helping out?"),
    ("landlord/roommate business message",
     "Hey, just a reminder that rent is due Friday, let me know if you need to Venmo or e-transfer this time"),
    ("genuine reconnection, no pitch",
     "hey it's been so long! how have you been, we should catch up sometime"),
]

for label, msg in hard_negatives:
    result = predict_mlm(msg)
    print(f"[{label}]")
    print(f"  text: {msg[:70]}...")
    print(f"  -> probability={result['probability']}, flag={result['flag']}")
    print(f"  -> top reasons: {result['top_reasons']}")
    print()

[legit recruiter, named role/company]
  text: Hi, I'm a recruiter at Google and came across your profile — we have a...
  -> probability=0.318, flag=False
  -> top reasons: [('team', 0.938), ('open', 0.657), ('profile', 0.288), ('hi', 0.266), ('came', 0.172)]

[friend selling own product, no recruiting]
  text: hey! I just started baking custom cakes for orders, would you want one...
  -> probability=0.189, flag=False
  -> top reasons: [('hey', 0.531), ('want', 0.518), ('just', 0.255), ('started', 0.135)]

[friend selling own craft business]
  text: hi! I opened an Etsy shop selling handmade candles, here's the link if...
  -> probability=0.252, flag=False
  -> top reasons: [('selling', 0.54), ('want', 0.502), ('hi', 0.426), ('link', 0.332), ('opened', 0.043)]

[fundraiser invite]
  text: Hey, a few of us are doing a charity 5k next month to raise money for ...
  -> probability=0.855, flag=True
  -> top reasons: [('team', 1.096), ('join', 1.062), ('join team', 0.626), ('money', 0.546),

In [13]:
train_mask = ~((df["source"] == "reddit") & (df["type"] == "dm"))
test_mask = (df["source"] == "reddit") & (df["type"] == "dm")

X_train_gen = df.loc[train_mask, "text"]
Y_train_gen = df.loc[train_mask, "label"]
X_test_gen = df.loc[test_mask, "text"]

print(f"train: {len(X_train_gen)} rows ({Y_train_gen.sum()} positive, all post/script-type)")
print(f"test:  {len(X_test_gen)} rows (all DM-type positives)")

vectorizer_gen = TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words="english")
X_train_gen_vec = vectorizer_gen.fit_transform(X_train_gen)
X_test_gen_vec = vectorizer_gen.transform(X_test_gen)

model_gen = LogisticRegression(class_weight="balanced", max_iter=1000)
model_gen.fit(X_train_gen_vec, Y_train_gen)

dm_proba = model_gen.predict_proba(X_test_gen_vec)[:, 1]
dm_caught = (dm_proba >= THRESHOLD).sum()

print(f"\nDMs caught at threshold {THRESHOLD}: {dm_caught} / {len(X_test_gen)} (recall={dm_caught/len(X_test_gen):.2f})")
print(f"DM probability distribution: min={dm_proba.min():.2f}, median={np.median(dm_proba):.2f}, max={dm_proba.max():.2f}")

train: 5248 rows (79 positive, all post/script-type)
test:  50 rows (all DM-type positives)

DMs caught at threshold 0.6: 23 / 50 (recall=0.46)
DM probability distribution: min=0.02, median=0.50, max=0.98
